In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a movie summarizer'),
    ('human', 'Please summarize the movie in brief : {input}')
])

In [3]:
#TASK -2

llm = init_chat_model(model="ollama:gemma3:4b")

In [4]:
#TASK - 3 :
str_parser = StrOutputParser()

## Parallel chain 1

In [5]:
# TASK  - 4 :
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text : str) -> dict:
    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

In [6]:
#TASK -1 # create prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Linkedin Post Generator'),
    ('human', 'Create a post for the following text for Linkedin : {text}')
])

#TASK -2

llm = init_chat_model(model="ollama:gemma3:4b")

#TASK - 3 :
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser


## Parallel chain 2

In [7]:
def insta_chain(text : dict):

    text = text["text"]

    insta_prompt = ChatPromptTemplate.from_messages([
        ('system','You are a instagram post generator'),
        ('human', 'Create a post for the following text for Instagram : {text}')
    ])

    llm = init_chat_model(model="ollama:gemma3:4b")

    str_parser = StrOutputParser()

    insta_chain = insta_prompt | llm | str_parser

    result  = insta_chain.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)


## Final orchestration

In [8]:
from langchain_core.runnables import RunnableParallel

In [9]:
final_chain = (
                prompt_template |
                llm |
                str_parser |
                dictionary_maker_runnable |
                RunnableParallel(branches={'linkedin' : chain_linkedin, 'instagram' : insta_chain_runnable})
)

In [10]:
final_chain.invoke("KGF")

{'branches': {'linkedin': 'Okay, here are a few LinkedIn post options based on your text, ranging in tone and length – pick the one that best suits your brand and audience!\n\n**Option 1 (Concise & Engaging - Great for quick engagement):**\n\n**(Image: A dynamic shot from K.G.F: Chapter 1 - ideally a powerful fight scene)**\n\n“Obsessed with the adrenaline-fueled world of K.G.F: Chapter 1! 🔥 This high-octane action epic about ambition, revenge, and a man fighting for his birthright is seriously captivating. Rocky’s journey is one you won’t soon forget. #KGF #Rocky #ActionMovie #Bollywood #Revenge #KingOfFighters”\n\n**Option 2 (Slightly More Detailed - Good for sparking conversation):**\n\n**(Image: A close-up of Yash as Rocky)**\n\n“Just finished revisiting K.G.F: Chapter 1 and I’m completely blown away! It’s the incredible story of Rocky – a young man who dreams of becoming a king, battling brutal training, shocking betrayals, and ultimately, becoming ‘K.G.F.’.  It’s a masterclass in

## Chain as RUnnable

In [11]:
# TASK [beautify dictinary]

def beautify_final_dict(final_response : dict ) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    insta_response = final_response['branches']['instagram']

    return {'linkedin' : linkedin_response, 'instagram' : insta_response}

beautify_final_dict_runnable  = RunnableLambda(beautify_final_dict)


# TASK 2 final chain

beautify_chain = final_chain | beautify_final_dict_runnable
beautify_chain

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a movie summarizer'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='Please summarize the movie in brief : {input}'), additional_kwargs={})])
| ChatOllama(output_version=None, model='gemma3:4b')
| StrOutputParser()
| RunnableLambda(dictionary_maker)
| {
    branches: {
                linkedin: ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a Linkedin Post Generator'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, 

In [12]:
beautify_chain.invoke("pushpa")

{'linkedin': "Okay, here are a few LinkedIn post options based on your text about *Pushpa: The Rise*, with varying lengths and tones – let me know which one you like best or if you’d like me to tweak them!\n\n**Option 1 (Concise & Engaging - Best for quick engagement):**\n\n**(Image: A striking image from Pushpa: The Rise - perhaps Pushpa looking determined)**\n\n“Absolutely gripping storytelling! 🤯 Just finished watching *Pushpa: The Rise* and it’s a masterclass in character development and suspense. Pushpa’s journey from a hopeful farmer to a key player in a dangerous underworld is captivating. Corruption, revenge, and a desperate fight for survival – this movie delivers it all! #PushpaTheRise #Bollywood #ActionMovie #Leadership #Resilience”\n\n**Option 2 (Slightly more detailed – Good for sparking conversation):**\n\n**(Image: A dynamic scene from the movie)**\n\n“Just experienced the incredible *Pushpa: The Rise* and I’m still thinking about Pushpa Raj Singh’s transformation! This 

## Conditional chains

In [15]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flat: Literal["positive", "negative", "neutral"]

llm_structured_output = llm.with_structured_output(llm_schema)
llm_structured_output.invoke('Movie is 50% good 50% bad')

llm_schema(movie_summary_flat='neutral')

In [ ]:
#conditional chains here
